# Week 91, Day 4: Documentation

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w091d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('waymark_docs.py', 'w', encoding='utf-8') as f:
    f.write('"""Walking times and routes for Waymark.\n\n>>> naismith_minutes(5, 0)\n60\n"""\n\n\ndef naismith_minutes(km, climb_m):\n    """Estimate walking minutes with Naismith\'s rule.\n\n    Args:\n        km: distance in kilometres.\n        climb_m: total climb in metres.\n\n    >>> naismith_minutes(6.0, 270)\n    99\n    """\n    return round(km / 5 * 60 + climb_m / 600 * 60)\n\n\ndef pace_ratio(actual, estimate):\n    """How an actual time compares with the estimate.\n\n    Args:\n        actual: minutes taken.\n        estimated: minutes from the rule.\n\n    >>> pace_ratio(90, 100)\n    0.9\n    >>> pace_ratio(120, 100)\n    1.25\n    """\n    return round(actual / estimate, 2)\n\n\ndef format_minutes(minutes):\n    return f"{minutes // 60} h {minutes % 60:02d} min"\n\n\nclass Route:\n    """A planned route: its stops, distance and climb."""\n\n    def __init__(self, stops, km, climb_m):\n        self.stops, self.km, self.climb_m = list(stops), km, climb_m\n\n    @property\n    def minutes(self):\n        """Estimated minutes for the route.\n\n        >>> Route(["Mill", "Weir"], 1.5, 15).minutes\n        20\n        """\n        return naismith_minutes(self.km, self.climb_m)\n\n    def describe(self):\n        return f"{\' to \'.join(self.stops)}: {self.km} km"\n\n    def _key(self):\n        return tuple(self.stops)\n')

**Predict before you run:** what will this print?



In [ ]:
def pace_ratio(actual, estimate):
    """How an actual time compares with the estimate.

    >>> pace_ratio(120, 100)
    1.25
    """
    return round(actual / estimate, 2)

print(pace_ratio(120, 100))

> **Key idea.** Sort documentation by the reader's need: tutorials, how-to guides, reference, explanation. Give the README what a newcomer looks for (what, install, quick start, usage, contributing, licence) and check it like code. Check docstrings against parameters with the syntax tree, and run their examples with doctest. Generate the reference from `inspect.signature` and `inspect.getdoc`.

### Exercise: Checking the README

Write `readme_problems(text)`: return a list of `"missing section: NAME"` for each of `Install`, `Quick start`, `Usage` and `Licence` (in that order) with no heading line (a line starting with `#`) whose text matches it, ignoring case; then `"unclosed code block"` if the number of lines starting with three backticks is odd; then `"TODO on line N"` for each line containing `TODO`.

In [ ]:
def readme_problems(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("FENCE = \"`\" * 3\n\ndef test_good():\n    \"a complete README\"\n    text = f\"# Waymark\\n\\nPlans walks.\\n\\n## Install\\n\\n{FENCE}\\npip install .\\n{FENCE}\\n\\n## Quick start\\n\\n## Usage\\n\\n## Licence\\n\\nMIT\\n\"\n    assert readme_problems(text) == []\n\ndef test_problems():\n    \"what a newcomer can't find\"\n    text = f\"# Waymark\\n\\n## install\\n\\n{FENCE}bash\\npip install .\\n\\n## Usage\\n\\nTODO: explain --steep\\n\"\n    assert readme_problems(text) == [\"missing section: Quick start\", \"missing section: Licence\", \"unclosed code block\", \"TODO on line 10\"]")

### Exercise: Docstrings against the code

Write `docstring_report(source)`, checking public (not starting with `_`) top-level functions and classes, and the public methods of those classes (named `Class.method`). Report `"LINE NAME: no docstring"` for each without one. For a function whose docstring has an `Args:` line, the documented parameters are the following lines that match an indented `name:`, up to the first blank line; report `"LINE NAME: undocumented parameter P"` for each parameter (other than `self` and `cls`) not documented, and `"LINE NAME: documents missing parameter P"` for each documented one that isn't a parameter. Sort by line, then by message.

In [ ]:
import ast
import re

def docstring_report(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_module():\n    \"waymark_docs.py\"\n    assert docstring_report(open(\"waymark_docs.py\").read()) == [\n        \"21 pace_ratio: documents missing parameter estimated\",\n        \"21 pace_ratio: undocumented parameter estimate\",\n        \"36 format_minutes: no docstring\",\n        \"55 Route.describe: no docstring\",\n    ]\n\ndef test_clean():\n    \"nothing to report\"\n    src = 'def f(a, b):\\n    \"\"\"Add.\\n\\n    Args:\\n        a: one.\\n        b: two.\\n    \"\"\"\\n    return a + b\\n\\ndef _private():\\n    pass\\n'\n    assert docstring_report(src) == []")

### Exercise: Examples that run

`format_minutes` has no docstring. Give it one: a summary line, then **at least two** doctest examples (include one under an hour), all passing. Don't change what it does.

In [ ]:
def format_minutes(minutes):
    return f"{minutes // 60} h {minutes % 60:02d} min"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import doctest\n\ndef test_examples():\n    \"documented and run\"\n    assert format_minutes(125) == \"2 h 05 min\" and format_minutes(45) == \"0 h 45 min\"\n    tests = doctest.DocTestFinder().find(format_minutes, globs={\"format_minutes\": format_minutes})\n    result = doctest.DocTestRunner().run(tests[0], out=lambda s: None)\n    assert result.attempted >= 2, \"at least two examples\"\n    assert result.failed == 0, \"every example passes\"\n    assert format_minutes.__doc__.strip().splitlines()[0].endswith(\".\"), \"a one-line summary ending in a full stop\"")

### Exercise: Reference from the code

Write `reference(objects)`: for each function or class, a heading `### NAME(SIGNATURE)` (from `inspect.signature`, which leaves out `self` for a class) and, on the next line, the first line of its cleaned docstring (`inspect.getdoc`), or `(no description)`. Separate entries with a blank line and return the text.

In [ ]:
import inspect

def reference(objects):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import waymark_docs as wd\n\ndef test_reference():\n    \"generated, so never out of date\"\n    assert reference([wd.naismith_minutes, wd.format_minutes, wd.Route]) == (\n        \"### naismith_minutes(km, climb_m)\\nEstimate walking minutes with Naismith's rule.\\n\\n\"\n        \"### format_minutes(minutes)\\n(no description)\\n\\n\"\n        \"### Route(stops, km, climb_m)\\nA planned route: its stops, distance and climb.\")\n    def plan(start, end, steep_limit=80):\n        \"\"\"\n        Plan the cheapest route.\n        \"\"\"\n    assert reference([plan]) == \"### plan(start, end, steep_limit=80)\\nPlan the cheapest route.\"")

### Exercise: Running the README

Write `run_examples(text, globs)`: find the `>>>` examples anywhere in `text` (with `doctest.DocTestParser`), run them with a copy of `globs`, and return `(attempted, failed)`, printing nothing.

In [ ]:
import doctest

def run_examples(text, globs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import waymark_docs as wd\n\nREADME = \"\"\"# Waymark\n\n## Quick start\n\n>>> naismith_minutes(6.0, 270)\n99\n>>> format_minutes(99)\n'1 h 39 min'\n\n## Usage\n\n>>> naismith_minutes(10, 0)\n100\n\"\"\"\n\ndef test_run():\n    \"the README, tested\"\n    globs = {\"naismith_minutes\": wd.naismith_minutes, \"format_minutes\": wd.format_minutes}\n    assert run_examples(README, globs) == (3, 1)\n    assert run_examples(\"No examples here.\", globs) == (0, 0)\n    assert \"x\" not in globs\n    assert run_examples(\">>> x = 2\\n>>> x * 3\\n6\\n\", globs) == (2, 0) and \"x\" not in globs")

### Bug hunt: The example that was wrong

The code is right: 270 m of climbing adds 27 minutes. The docstring's second example says 30, so doctest reports `attempted 2, failed 1`. Fix the **example** so it prints `attempted 2, failed 0`.

In [ ]:
import doctest

def climb_minutes(climb_m):
    """Minutes added for climbing, by Naismith's rule.

    >>> climb_minutes(600)
    60
    >>> climb_minutes(270)
    30
    """
    return round(climb_m / 600 * 60)

test = doctest.DocTestFinder().find(climb_minutes, globs={"climb_minutes": climb_minutes})[0]
result = doctest.DocTestRunner().run(test, out=lambda s: None)
print(f"attempted {result.attempted}, failed {result.failed}")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"documentation that tells the truth\"\n    expect_output(\"attempted 2, failed 0\")\n    assert climb_minutes(270) == 27, \"fix the example, not the code\"")

### Bug hunt: A summary that came out blank

The docstring starts on the line after the quotes, so the first line of `__doc__` is empty and this prints `plan: `. Take the summary from the cleaned docstring, so it prints `plan: Plan the shortest route.`

In [ ]:
import inspect

def plan(start, end):
    """
    Plan the shortest route.

    Returns a Route.
    """

def summary(fn):
    return fn.__doc__.split("\n")[0]

print(f"{plan.__name__}: {summary(plan)}")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"inspect.getdoc\"\n    expect_output(\"plan: Plan the shortest route.\")")

### Puzzle: Generating the reference

Arrange `reference`: a heading with the signature, and the summary from the cleaned docstring, for each object. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the Diátaxis kind for "How to import a map from CSV" (one word, with a hyphen)
q = ___   # examples doctest attempts in a docstring with three >>> lines (each with its output)
r = ___   # str(inspect.signature(lambda a, b=2: 0))
s = ___   # True or False: docstring_report checks the method _key
t = ___   # the number of the PEP on docstring conventions

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"how-to\" and q == 3 and r == \"(a, b=2)\", \"p, q and r\"\n    assert s is False and t == 257, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the documentation report

Write `docs_report(source, readme)`, using `docstring_report`-style rules and doctest. Public items are top-level functions and classes not starting with `_`, and their methods not starting with `_`. Return a list of lines:

1. `"docstrings: D of P public items (PCT%)"`, with PCT rounded to a whole number (100 if there are none);
2. `"examples: A run, F failing"`: every `>>>` example in the module's docstring and in the docstrings of **all** functions, classes and methods (public or not), run in a namespace made by executing `source`;
3. `"readme: ok"` or `"readme: missing "` and the missing sections of `Install`, `Quick start`, `Usage`, `Licence` (headings compared ignoring case), joined by `", "`;
4. if any public item lacks a docstring, `"undocumented: "` and their names in source order (methods as `Class.method`), joined by `", "`;
5. finally `"docs ready"` if every public item is documented, no example fails and the README is complete, otherwise `"docs need work"`.

In [ ]:
import ast
import doctest

def docs_report(source, readme):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("README = \"# Waymark\\n\\n## Install\\n\\n## Usage\\n\\n## Licence\\n\"\n\ndef test_module():\n    \"waymark_docs.py\"\n    assert docs_report(open(\"waymark_docs.py\").read(), README) == [\n        \"docstrings: 4 of 6 public items (67%)\",\n        \"examples: 5 run, 1 failing\",\n        \"readme: missing Quick start\",\n        \"undocumented: format_minutes, Route.describe\",\n        \"docs need work\",\n    ]\n\ndef test_ready():\n    \"everything documented and passing\"\n    src = 'def double(x):\\n    \"\"\"Double it.\\n\\n    >>> double(4)\\n    8\\n    \"\"\"\\n    return 2 * x\\n\\ndef _helper():\\n    \"\"\"\\n    >>> 1 + 1\\n    2\\n    \"\"\"\\n'\n    full = README + \"\\n## Quick start\\n\"\n    assert docs_report(src, full) == [\"docstrings: 1 of 1 public items (100%)\", \"examples: 2 run, 0 failing\", \"readme: ok\", \"docs ready\"]")

## Warm-up

The docstring promises 1.25; the code says 1.2. A reader who trusts the example is misled, and nothing complains. An example that's **run** as a test can't drift like that.

## Four kinds of documentation

The **Diátaxis** framework (by Daniele Procida) sorts documentation by what the reader is doing:

- **Tutorials** teach by doing: "Plan your first walk", a guided lesson that always works.
- **How-to guides** solve one problem for someone who knows the basics: "How to import a map from CSV".
- **Reference** describes the machinery, completely and drily: every command, option, function and API route.
- **Explanation** discusses why: "Why Waymark uses Naismith's rule", "How round walks are found".

Mixing them is the usual failure: a reference page that wanders into a tutorial, or a tutorial that stops to list every option. For a capstone, a README with a quick start, two or three how-to guides, a generated reference and one explanation page is plenty.

## The README

The README is the front door. A newcomer looks for, in order: **what it is** (one paragraph and perhaps a screenshot), how to **install** it, a **quick start** that works when copied, **usage** for the common tasks, how to **contribute** and run the tests, and the **licence**. Check it as you'd check code: every heading present, every code block closed, no `TODO` left in, and every command in it actually run (the examples below, and Week 88's self-checking demo).

## Docstrings that can't drift

Docstrings follow PEP 257: a one-line summary ending in a full stop, a blank line, then details. A common style for parameters is an `Args:` section with one indented `name: description` line each. Two kinds of check keep them honest:

- **Against the code**: the syntax tree knows each function's parameters, so a script can report public functions without docstrings, parameters missing from `Args:`, and documented parameters that no longer exist.
- **Against the behaviour**: **doctest** finds `>>>` examples in docstrings (or any text, such as a README), runs them, and compares the output. `python -m doctest README.md -v` on your own computer, or `pytest --doctest-modules`, makes them part of the test suite.

Keep doctest examples small and deterministic: they're documentation first.

## Reference from the code

Reference should be **generated**, so it can't fall behind: `inspect.signature(fn)` gives the parameters with defaults, and `inspect.getdoc(fn)` the docstring with its indentation cleaned. Week 90's API registry did the same for routes. Tools such as Sphinx and MkDocs (both free) build whole sites from docstrings on your own computer.

## Practice

## Quiz

## Daily challenge

## Your capstone today

1. **Sort what you have** into the four kinds, and write what's missing: at least a quick start, two how-to guides and one explanation.
2. **Run the README**: every command and every `>>>` example, in CI (`pytest --doctest-modules`, and doctest on the README).
3. **Fix the drift** `docstring_report` finds in your code.
4. **Generate the reference** with Sphinx, MkDocs or your own script, and link it from the README.

## Before you go

Tomorrow you'll watch real people use your capstone, which is the fastest way to find out which of today's documents they actually read.